# Optimizing Ride Booking Conversion: Funnel Analysis for a Mobility App

### SwiftRide Product Analytics Case Study
Identifying drop-off points and behavioral patterns across the ride-booking journey.

## 1. Business Problem

SwiftRide is a fictional ride-hailing company experiencing declining ride completion rates over the last quarter.

The product analytics team is tasked with identifying major funnel drop-off points, understanding user behavior across segments, and evaluating potential product improvements.

### Key Questions
1. What is the overall ride-booking funnel conversion rate?
2. Which funnel stage has the highest drop-off?
3. Which user segments are most affected?
4. Do city and device type influence conversion?
5. Can product experiments improve conversion?

## 2. Dataset Schema

This project uses three relational tables:

### users.csv
Contains user-level attributes such as city, device type, and segment.

### ride_sessions.csv
Contains session-level booking activity.

### ride_events.csv
Contains event-level funnel logs tracking the ride-booking journey.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Load datasets
events_df = pd.read_csv("../data/raw/ride_events.csv")
sessions_df = pd.read_csv("../data/raw/ride_sessions.csv")
users_df = pd.read_csv("../data/raw/users.csv")

## 3. Data Validation

Validate dataset dimensions, schema consistency, and event distributions before analysis.

In [ ]:
events_df.shape, sessions_df.shape, users_df.shape

((2003353, 5), (429119, 6), (50000, 6))

In [ ]:
users_df.head()
sessions_df.head()
events_df.head()

,event_id,session_id,user_id,event_name,event_time
0,E00000001,S0000001,U000001,app_open,2026-03-31 17:42:00
1,E00000002,S0000001,U000001,destination_selected,2026-03-31 17:42:30
2,E00000003,S0000001,U000001,fare_shown,2026-03-31 17:43:00
3,E00000004,S0000001,U000001,ride_requested,2026-03-31 17:43:30
4,E00000005,S0000001,U000001,driver_assigned,2026-03-31 17:44:00


In [ ]:
events_df["event_name"].value_counts()

event_name
app_open                429119
destination_selected    391508
fare_shown              379500
ride_requested          298113
driver_assigned         258528
ride_completed          246585
Name: count, dtype: int64

### Initial Validation Summary

- Dataset successfully loaded
- All three tables have expected dimensions
- Funnel event hierarchy appears valid
- Event volume decreases progressively across funnel stages

## 4. Funnel Analysis

Analyze overall funnel conversion from app open to ride completion.

### Session Count by Funnel Stage

In [ ]:
# Count sessions per event
funnel_counts = (
    events_df.groupby("event_name")["session_id"]
    .nunique()
    .reset_index(name="sessions")
)

funnel_counts

,event_name,sessions
0,app_open,429119
1,destination_selected,391508
2,driver_assigned,258528
3,fare_shown,379500
4,ride_completed,246585
5,ride_requested,298113


In [ ]:
# Reorder funnel steps to match booking journey
funnel_order = [
    "app_open",
    "destination_selected",
    "fare_shown",
    "ride_requested",
    "driver_assigned",
    "ride_completed"
]

funnel_counts["event_name"] = pd.Categorical(
    funnel_counts["event_name"],
    categories=funnel_order,
    ordered=True
)

funnel_counts = (
    funnel_counts
    .sort_values("event_name")
    .reset_index(drop=True)
    .copy()
)

funnel_counts

,event_name,sessions
0,app_open,429119
1,destination_selected,391508
2,fare_shown,379500
3,ride_requested,298113
4,driver_assigned,258528
5,ride_completed,246585


### Funnel Conversion Metrics

In [ ]:
# Step conversion
funnel_counts["step_conversion"] = (
    funnel_counts["sessions"] /
    funnel_counts["sessions"].shift(1)
)

# Overall conversion
total_sessions = funnel_counts.iloc[0]["sessions"]

funnel_counts["overall_conversion"] = (
    funnel_counts["sessions"] / total_sessions
)

# Drop-off rate
funnel_counts["dropoff_rate"] = 1 - funnel_counts["step_conversion"]

# Convert to percentage
funnel_counts["step_conversion"] = (
    funnel_counts["step_conversion"] * 100
).round(2)

funnel_counts["overall_conversion"] = (
    funnel_counts["overall_conversion"] * 100
).round(2)

funnel_counts["dropoff_rate"] = (
    funnel_counts["dropoff_rate"] * 100
).round(2)

funnel_counts



,event_name,sessions,step_conversion,overall_conversion,dropoff_rate
0,app_open,429119,NaN,100.00,NaN
1,destination_selected,391508,91.24,91.24,8.76
2,fare_shown,379500,96.93,88.44,3.07
3,ride_requested,298113,78.55,69.47,21.45
4,driver_assigned,258528,86.72,60.25,13.28
5,ride_completed,246585,95.38,57.46,4.62


### Funnel Visualization

In [ ]:
# Funnel Session Count Chart

from matplotlib.ticker import FuncFormatter

# stage labels
stage_labels = {
    "app_open": "App Open",
    "destination_selected": "Destination Selected",
    "fare_shown": "Fare Shown",
    "ride_requested": "Ride Requested",
    "driver_assigned": "Driver Assigned",
    "ride_completed": "Ride Completed"
}

# Copy to avoid modifying original table
plot_df = funnel_counts.copy()
plot_df["stage_label"] = plot_df["event_name"].map(satge_labels)

plt.figure(figsize=(10,5))

plt.barh(
    plot_df["stage_label"],
    plot_df["sessions"]
)

# Add value labels at end of bars
for index, value in enumerate(plot_df["sessions"]):
    plt.text(value + 3000, index, f"{value:,}", va="center")

# Format x-axis
plt.gca().xaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f"{int(x):,}")
)

plt.xlabel("Number of Sessions")
plt.ylabel("Funnel Stage")
plt.tittle("Ride Booking Funnel by Session Count")

# Make top stage appear on top
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()


### Funnel Analysis Insights

Key findings from the overall funnel analysis:

- Overall ride completion rate is **57.46%**, meaning only about 57 out of 100 booking sessions result in completed rides.
- The largest drop-off occurs between **fare_shown** and **ride_requested** with a drop-off rate of **21.45%**.
- This suggests users are highly sensitive to pricing during booking confirmation.
- A secondary friction point occurs between **ride_requested** and **driver_assigned** with a **13.28%** drop-off, indicating potential driver supply issues.
- Early funnel stages remain healthy, suggesting the main friction happens during pricing and fulfillment stages.

## 5. Segment Analysis

Compare funnel performance across user segments.

## 6. Device and City Analysis

Analyze whether device type or city affects booking conversion.

## 7. Experiment Analysis

Evaluate conversion differences between control and treatment groups.

## 8. Statistical Testing

Perform hypothesis testing to evaluate experiment significance.

## 9. Final Business Recommendation

In [ ]:
# sessions_df["user_segment"].value_counts()

In [ ]:
# # 1. ambil semua session yang completed
# completed_sessions = set(
#     events_df.loc[
#         events_df["event_name"] == "ride_completed",
#         "session_id"
#     ]
# )

# # 2. copy sessions table
# segment_eval = sessions_df.copy()

# # 3. flag completed
# segment_eval["is_completed"] = (
#     segment_eval["session_id"]
#     .isin(completed_sessions)
# )

# # 4. summary
# segment_result = (
#     segment_eval
#     .groupby("user_segment")
#     .agg(
#         total_sessions=("session_id", "count"),
#         completed_sessions=("is_completed", "sum")
#     )
#     .reset_index()
# )

# segment_result["completion_rate"] = (
#     segment_result["completed_sessions"]
#     / segment_result["total_sessions"]
#     * 100
# ).round(2)

# segment_result